# Verify C1/A1 demo parity after completed calibration
No rescoring of full calibration splits. No threshold tuning or publication.
Attach the original ASVspoof audio dataset and latest final calibration backup. This notebook includes the exact local source bundle, so no source push is needed. Use a fresh session. It creates a separate pinned CPU verification environment, stages saved scores, compares 256 deterministic test recordings per model plus 256 clean-validation cache records, and exports the reports.
Legacy test CSVs are not used. Short clips are counted as expected rejections; other rejections fail verification. Download the final verification ZIP and return it to this chat. Deployment remains blocked pending parity and final local checks.


In [ ]:
REPO_URL = "https://github.com/Remigaraki/TwoStream-Audio-Forensics.git"
BRANCH = "main"
DATA_ROOT = "/kaggle/input/datasets/ulianazb/asvspoof-2024"
REPO = "/kaggle/tmp/TwoStream-Demo-Verification"
MANIFEST = "/kaggle/working/manifest.csv"
BACKUP = None  # Set to the final ZIP path OR its extracted root containing demo_thresholds.json.
BACKUP_NAME = "calibration_backup_20261006_140731_113645.zip"
OUT = "/kaggle/working/demo_verification"


In [ ]:
import os, sys, subprocess, shutil, base64, io, zipfile, json
from pathlib import Path

def run(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)), flush=True)
    p = subprocess.Popen(list(map(str, cmd)), cwd=cwd or REPO, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout: print(line, end="", flush=True)
    if p.wait(): raise RuntimeError(f"Command failed: exit {p.returncode}")

Path(REPO).parent.mkdir(parents=True, exist_ok=True)
if not (Path(REPO) / ".git").is_dir():
    if Path(REPO).exists(): raise RuntimeError("Existing non-checkout directory; inspect before cloning")
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO],
                   env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}, check=True)
if subprocess.run(["git", "lfs", "version"], capture_output=True).returncode != 0:
    run(["apt-get", "-qq", "update"], cwd="/")
    run(["apt-get", "-qq", "install", "-y", "git-lfs"], cwd="/")
run(["git", "lfs", "install", "--local"])
run(["git", "lfs", "pull", "--include",
     "checkpoints/setup_c/best_c1_attention.pt,checkpoints/setup_a/best_a_ep13_eer0105_0713_0401.pt"])


In [ ]:
# Exact source bundle from the local verified implementation.
SOURCE_BUNDLE = "..."
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_BUNDLE))) as z:
    for name in z.namelist():
        target = Path(REPO) / name
        if not target.resolve().is_relative_to(Path(REPO).resolve()): raise RuntimeError("Unsafe source path")
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(z.read(name))
print("Applied source bundle. No thresholds installed.")


In [ ]:
# Separate CPU environment matches the pinned local demo runtime.
ENV_ROOT = "/kaggle/tmp/demo_verify_env"
run([sys.executable, "-m", "venv", "--without-pip", "--system-site-packages", ENV_ROOT], cwd="/kaggle/tmp")
PYTHON = str(Path(ENV_ROOT) / "bin/python")
run([sys.executable, "-m", "pip", "--python", PYTHON, "install", "torch==2.11.0", "torchaudio==2.11.0",
     "--index-url", "https://download.pytorch.org/whl/cpu"])
run([sys.executable, "-m", "pip", "--python", PYTHON, "install", "-r", "demo/requirements.txt",
     "--extra-index-url", "https://download.pytorch.org/whl/cpu", "pydub", "pandas", "matplotlib"])
# Build paths for this session rather than restore stale manifest locations.
if not Path(MANIFEST).exists():
    run([PYTHON, "scripts/build_manifest.py", "--input_root", DATA_ROOT, "--output", MANIFEST])
run([PYTHON, "scripts/verify_manifest.py", "--manifest", MANIFEST])


In [ ]:
if BACKUP is None:
    matches = list(Path("/kaggle/input").rglob(BACKUP_NAME))
    if len(matches) == 1:
        BACKUP = str(matches[0])
    else:
        roots = [p.parent for p in Path("/kaggle/input").rglob("demo_thresholds.json")
                 if (p.parent / "calibration/c1_corrected_fb77a9d19bfb/val_C1_3047175ddc1b_aac_128.csv").exists()]
        if len(roots) != 1: raise RuntimeError("Set BACKUP to the final ZIP or its extracted root")
        BACKUP = str(roots[0])
backup_path = Path(BACKUP)
if backup_path.is_dir():
    # Content identity is stable even when Kaggle automatically extracted the ZIP.
    packed = Path("/kaggle/working/restored_input_backup.zip")
    with zipfile.ZipFile(packed, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(backup_path.rglob("*")):
            if p.is_file(): z.write(p, p.relative_to(backup_path).as_posix())
    backup_path = packed
run([PYTHON, "demo/stage_calibration.py", "--backup", str(backup_path), "--output", OUT])


In [ ]:
for model in ["C1", "A1"]:
    run([PYTHON, "demo/check_parity.py", "--model", model, "--manifest", MANIFEST,
         "--limit", "256", "--evaluation_device", "cpu",
         "--thresholds", str(Path(OUT) / "thresholds.candidate.json"),
         "--calibration_cache", str(Path(OUT) / "cache" / model),
         "--report", str(Path(OUT) / f"parity_{model}.json")])
print("Both reports passed. Active thresholds are still unchanged.")


In [ ]:
# Run this export cell even if parity failed, so the failure can be investigated.
from IPython.display import FileLink, display
result_zip = Path("/kaggle/working/demo_parity_verification.zip")
with zipfile.ZipFile(result_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(Path(OUT).rglob("*")):
        if p.is_file(): z.write(p, p.relative_to(OUT).as_posix())
    z.writestr("verification_source_bundle.zip", base64.b64decode(SOURCE_BUNDLE))
with zipfile.ZipFile(result_zip) as z: assert z.testzip() is None
print("Download this ZIP and return it for local validation and threshold installation.")
display(FileLink(str(result_zip)))
